In [ ]:
import sys
import json
from pathlib import Path
sys.path.append(str(Path("../").resolve()))


import pandas as pd
import numpy as np


import matplotlib as plt
import seaborn as sb


from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold


from sklearn.linear_model import LogisticRegression


from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import classification_report
from sklearn.metrics import roc_auc_score
from sklearn.metrics import fbeta_score
from sklearn.metrics import make_scorer


from sklearn.pipeline import Pipeline
from sklearn.pipeline import make_pipeline


from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.feature_selection import RFE


from src.prerocessing import get_scalar, scale_df
from src.feature_selection import *


1) Load JSON
2) Feature selection
3) Data Transform
4) Test-Train split
5) Training with specific model
6) Evaluate
7) Save evaluate vals.

## 1) Load JSON & df
- vsetky nastavenia su realizovane v conf.json, a teda ak chceme nieco menit v experimentalnej pipeline, treba zmenit `conf.json` a spustit ju

In [88]:
conf_path: str = "../../conf/conf.json"
df_path: str = "../../data/clean_ds.xlsx"

# --- dataframe ---
df = pd.read_excel(df_path)


# --- JSON ---
with open(conf_path, 'r') as file:
    data = json.load(file)

#print(json.dumps(data))

config: dict = data["config"][0]

### 1.1) Rozdelenie datasetu na atributya predikovanu hodnotu

In [89]:
X = df.drop('referred_CXL',
            axis=1
            )
y = df.referred_CXL

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=config["test_size"],
    random_state=config["random_state"]
    )

## 2) Data Tramsformation

In [90]:
# --- scale ---
scaler = get_scalar(config["scaling"])
#X_train_scaled, X_test_scaled = scale_df(X_train, X_test, scalar)


## 3) Pipeline

### 3.1) Aplikovanie vybrateho modelu

In [91]:
base_model = LogisticRegression(max_iter=config["epoch"],
                           random_state=config["random_state"],
                           class_weight="balanced"
                           )

### 3.2) Aplikovanie pipeline

In [ ]:
f2_score = make_scorer(fbeta_score, beta=2)

eval_metrics: dict = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "recall_weighted": "recall_weighted",
    "roc_auc": "roc_auc",
    "f2_score": f2_score
}


# "balanced"
class_weights: dict = {0: 1.5,
                       1: 1.0}

pipeline = Pipeline([
    ("scaler", scaler),
    ("feature_selection", RFE(
        estimator=base_model,
        n_features_to_select=config["n_features_to_selection"]
    )),
    ("classifier", LogisticRegression(
        C=0.1,                                  # regularizacia
        max_iter=config["epoch"],
        random_state=config["random_state"],
        class_weight="balanced"
    ))
])

#pipeline.fit(X_train, y_train)
results = cross_validate(pipeline, X, y, cv=10, scoring=eval_metrics)

/opt/anaconda3/envs/iau/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/opt/anaconda3/envs/iau/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preproces

## 4) Evaluacia

In [93]:
"""y_pred = pipeline.predict(X_test)
acc = accuracy_score(y_test, y_pred) * 100
prec = precision_score(y_test, y_pred) * 100
rec = recall_score(y_test, y_pred) * 100

print(f"model acc.: {acc:.2f}%")
print(f"model prec.: {prec:.2f}%")
print(f"model rec.: {rec:.2f}%")"""

'y_pred = pipeline.predict(X_test)\nacc = accuracy_score(y_test, y_pred) * 100\nprec = precision_score(y_test, y_pred) * 100\nrec = recall_score(y_test, y_pred) * 100\n\nprint(f"model acc.: {acc:.2f}%")\nprint(f"model prec.: {prec:.2f}%")\nprint(f"model rec.: {rec:.2f}%")'

In [ ]:
acc = results["test_accuracy"].mean() * 100
prec = results["test_precision"].mean() * 100
rec = results["test_recall"].mean() * 100
rec_weig = results["test_recall_weighted"].mean() * 100
roc_auc = results["test_roc_auc"].mean() * 100
f2_sc = results["test_f2_score"].mean() * 100

print(f"Accuracy: {acc:.2f}%")
print(f"Precision: {prec:.2f}%")
print(f"Recall: {rec:.2f}%")
print(f"Recall Weighted: {rec_weig:.2f}%")
print(f"ROC AUC: {roc_auc:.2f}%")
print(f"f2 score: {f2_sc:.2f}%")

Accuracy: 78.30%
Precision: 56.88%
Recall: 81.62%
Recall Weighted: 78.30%
ROC AUC: 87.67%


# TODO
- [x] ako funguje pipeline?
- [x] ako mozem aplikovat corss validation
- [] upravit decision threshold
- [x] pridaj ine metriky na evaliaciu

## Biznis model
- hladam chorobu = klucovy Recall
- potrebujem vysoky Recall (70-80) a rozumny Precision (cca 60)
- lepsie pacienta dodatocne vysetrit nez poslat choreho domov